# Phenotype Comparison

Runs several `ExperimentConfig` phenotypes (e.g. energy-conservative vs. exploratory vs.
high-prospective-depth) head-to-head through `System0Sandbox` and compares their
CES-like efficiency, survival rate, energy spend, and prospective-forecast accuracy.

This is a self-contained sweep — it runs the agents itself (deterministically, via seeded
RNG) rather than depending on previously-saved `results/` artifacts.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from metabolic_intelligence_lab.experiments.configs import ExperimentConfig
from metabolic_intelligence_lab.system0_sandbox import System0Sandbox

STEPS = 40
config_paths = sorted(Path("../configs").glob("*.yaml")) or sorted(Path("configs").glob("*.yaml"))
configs = [ExperimentConfig.from_yaml(p) for p in config_paths]
for cfg in configs:
    cfg.steps = STEPS
[c.name for c in configs]

In [ ]:
runs = {}
for cfg in configs:
    sb = System0Sandbox(cfg)
    sb.run(verbose=False)
    runs[cfg.name] = sb

rows = []
for name, sb in runs.items():
    s = dict(sb.summary)
    s["phenotype"] = name
    s["prospective_depth"] = sb.cfg.prospective_depth
    s["salience_threshold"] = sb.cfg.salience_threshold
    s["tool_budget"] = sb.cfg.tool_budget
    acc = sb.agent.plog.accuracy_summary()
    s["forecast_avg_score"] = acc["avg_score"]
    s["forecast_hit_rate"] = acc["hit_rate"]
    rows.append(s)

comparison = pd.DataFrame(rows).set_index("phenotype")
comparison

## CES-like efficiency & survival rate by phenotype

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
comparison["avg_ces_like"].plot(kind="bar", ax=axes[0], color="darkorange")
axes[0].set_title("Avg CES-like efficiency")
axes[0].set_ylabel("avg_ces_like")
axes[0].tick_params(axis="x", rotation=45)

comparison["survival_rate"].plot(kind="bar", ax=axes[1], color="seagreen")
axes[1].set_title("Survival rate")
axes[1].set_ylabel("survival_rate")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## Energy spend vs. forecast accuracy

Does deeper prospective look-ahead (`prospective_depth`) pay for itself in forecast
accuracy, or just burn more energy on System-2 escalation?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(comparison["avg_energy"], comparison["forecast_avg_score"],
                s=120, c=comparison["prospective_depth"], cmap="viridis")
for name, row in comparison.iterrows():
    ax.annotate(name, (row["avg_energy"], row["forecast_avg_score"]),
                textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.set_xlabel("avg_energy")
ax.set_ylabel("forecast_avg_score (Jaccard)")
ax.set_title("Energy spend vs. prospective-forecast accuracy")
fig.colorbar(sc, ax=ax, label="prospective_depth")
plt.tight_layout()
plt.show()

## Salience-entropy & stability across phenotypes

`entropy_score` reflects how spread the agent's attention is across labels (high = exploratory,
low = fixated); `stability_score` reflects how consistent its top-salience choice stays tick to tick.

In [ ]:
comparison[["entropy_score", "stability_score", "frontier_variance"]].plot(
    kind="bar", figsize=(9, 4), rot=45
)
plt.title("Attention dynamics by phenotype")
plt.tight_layout()
plt.show()